In [2]:
import pandas as pd
import pprint
import configPinecone
from pinecone import Pinecone, ServerlessSpec
import pinecone
from sentence_transformers import SentenceTransformer
import numpy as np

In [3]:
files = pd.read_csv("course_section_descriptions.csv", encoding='ANSI')

In [9]:
pc = Pinecone(api_key = configPinecone.pineconeDefaultAPIKey, environment = configPinecone.pineconeEnv)
pc.create_index(
    name = "weighted-semantic-search-index",
    dimension = 768,
    metric = "cosine",
    spec = ServerlessSpec(
        cloud = "aws",
        region = "us-east-1"
    )
)

{
    "name": "weighted-semantic-search-index",
    "metric": "cosine",
    "host": "weighted-semantic-search-index-cwx78x5.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 768,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
     

In [5]:
# Define weights for different text components
weight_course_name = 5
weight_section_name = 3
weight_section_description = 2
weight_other = 1  # For other components like course technology and course description

# for each section
files['unique_id'] = files['course_id'].astype(str) + '-' + files['section_id'].astype(str)
files['metadata'] = files.apply(lambda row: {
    "course_name": row['course_name'],
    "section_name": row['section_name'],
    "section_description": row['section_description']
}, axis=1)

In [6]:
def create_embeddings(row):
    # Encode individual components
    emb_course_name = model.encode(row['course_name'], show_progress_bar=False) * weight_course_name
    emb_section_name = model.encode(row['section_name'], show_progress_bar=False) * weight_section_name
    emb_section_description = model.encode(row['section_description'], show_progress_bar=False) * weight_section_description
    emb_course_tech = model.encode(row['course_technology'], show_progress_bar=False) * weight_other
    emb_course_desc = model.encode(row['course_description'], show_progress_bar=False) * weight_other

    # Combine embeddings by averaging them
    combined_embedding = (emb_course_name + emb_section_name + emb_section_description + emb_course_tech + emb_course_desc) / (weight_course_name + weight_section_name + weight_section_description + 2 * weight_other)
    return combined_embedding

model = SentenceTransformer('multi-qa-distilbert-cos-v1')
# Apply the function to create weighted embeddings
files['embedding'] = files.apply(create_embeddings, axis=1)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

D:\anaconda3\envs\llm\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\hf_cache\hub\models--sentence-transformers--multi-qa-distilbert-cos-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
index = pc.Index("weighted-semantic-search-index")

In [11]:
# Prepare the vectors for upserting
vectors_to_upsert = [(row['unique_id'], row['embedding'].tolist(), row['metadata']) for index, row in files.iterrows()]

In [12]:
index.upsert(vectors=vectors_to_upsert)
print("Data successfully upserted to Pinecone index.")

Data successfully upserted to Pinecone index.


In [13]:
query = "lasso"
query_embedding = model.encode(query, show_progress_bar=False).tolist()

In [14]:
query_results = index.query(
    vector=[query_embedding],
    top_k=15,
    include_metadata=True
)

In [15]:
score_threshold = 0.3

In [16]:
for match in query_results['matches']:
    if match['score'] >= score_threshold:
        course_details = match.get('metadata', {})
        course_name = course_details.get('course_name', 'N/A')
        section_name = course_details.get('section_name', 'N/A')
        section_description = course_details.get('section_description', 'No description available')
        
        pprint.pprint(f"Matched item ID: {match['id']}, Score: {match['score']}")
        pprint.pprint(f"Course: {course_name}")
        pprint.pprint(f"Section: {section_name}, Description: {section_description}", width = 100) 

'Matched item ID: 56-492, Score: 0.618946671'
'Course: Machine Learning with Ridge and Lasso Regression'
('Section: Ridge and Lasso Regression – Practical Case, Description: In this section, we will walk '
 'you through the implementation of ridge and lasso regression using sk-learn in Python. We apply '
 'these methods to a real dataset in order to increase the performance of a regression algorithm '
 'by preventing overfitting. Furthermore, we demonstrate how regularization works and uncover the '
 'differences between ridge and lasso models.  ')
'Matched item ID: 56-490, Score: 0.57380271'
'Course: Machine Learning with Ridge and Lasso Regression'
('Section: Regularization Basics, Description: As an introduction to the course, we explore the '
 'concept of regularization and explain how it can be leveraged to prevent overfitting and '
 'multicollinearity issues. In addition, we demonstrate the theoretical differences between the '
 'mechanisms of ridge and lasso regression. ')
'Matc